# Lesson 01 - Εισαγωγή στους Πράκτορες Τεχνητής Νοημοσύνης

Καλώς ήρθατε στο πρώτο μάθημα του μαθήματος **Πράκτορες Τεχνητής Νοημοσύνης για Αρχάριους**!

Ένας **πράκτορας ΤΝ** είναι ένα πρόγραμμα που χρησιμοποιεί ένα μεγάλο γλωσσικό μοντέλο (LLM) ως κινητήρια δύναμη λογικής και μπορεί να εκτελέσει *ενέργειες* στον πραγματικό κόσμο — καλώντας APIs, ερωτώντας βάσεις δεδομένων ή εκτελώντας κώδικα — προκειμένου να επιτύχει έναν στόχο εκ μέρους ενός χρήστη.

Σε αυτό το σημειωματάριο θα δημιουργήσετε τον πρώτο σας πράκτορα: έναν **Ταξιδιωτικό Πράκτορα** που προτείνει προορισμούς διακοπών. Καθώς προχωράτε, θα μάθετε πώς να:

1. Συνδεθείτε στην υπηρεσία <i>Azure AI Foundry Agent Service</i> χρησιμοποιώντας το **Microsoft Agent Framework**.
2. Δώσετε στον πράκτορα ένα **εργαλείο** — μια απλή συνάρτηση Python που μπορεί να καλέσει.
3. Εκτελέσετε τον πράκτορα και να ελέγξετε την απάντησή του.
4. Ρεύμα-μεταδώσετε την απάντηση του πράκτορα διακριτό-τις-λέξεις token.


## Ρύθμιση

Πριν εκτελέσετε αυτό το σημειωματάριο, βεβαιωθείτε ότι έχετε:

1. **Ένα έργο Azure AI Foundry** με αναπτυγμένο μοντέλο συνομιλίας (π.χ. `gpt-4o-mini`).
2. **Είστε συνδεδεμένοι με το Azure CLI** — εκτελέστε `az login` στο τερματικό σας.
3. **Ορίσατε τις απαιτούμενες μεταβλητές περιβάλλοντος:**
   - `AZURE_AI_PROJECT_ENDPOINT` — το σημείο τερματισμού του έργου Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — το όνομα του αναπτυγμένου μοντέλου σας.

Το κελί παρακάτω εγκαθιστά τα πακέτα Python που χρειάζεστε.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Δημιουργία του Πρώτου σας Πράκτορα

Ένας πράκτορας χρειάζεται δύο πράγματα:

- **Οδηγίες** που του λένε *ποιος είναι* και *πώς να συμπεριφέρεται* (μία προτροπή συστήματος).
- **Εργαλεία** — συναρτήσεις Python διακοσμημένες με `@tool` που ο πράκτορας μπορεί να καλέσει για να ανακτήσει πληροφορίες ή να εκτελέσει ενέργειες.

Παρακάτω ορίζουμε ένα απλό εργαλείο που επιστρέφει μια λίστα με δημοφιλείς προορισμούς διακοπών. Ο πράκτορας θα χρησιμοποιήσει αυτό το εργαλείο όταν ένας χρήστης ζητήσει προτάσεις ταξιδιών.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Για μια πιο διαδραστική εμπειρία μπορείτε να **μεταδώσετε** την απάντηση του πράκτορα. Αντί να περιμένετε την πλήρη απάντηση, ο πράκτορας παρέχει κομμάτια κειμένου καθώς παράγονται. Αυτό είναι ιδιαίτερα χρήσιμο σε διεπαφές συνομιλίας όπου θέλετε να εμφανίσετε την έξοδο σε πραγματικό χρόνο.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

Σε αυτό το μάθημα μάθατε πώς να:

- **Δημιουργήσετε έναν πάροχο** που συνδέεται με την υπηρεσία Azure AI Foundry Agent μέσω `AzureAIProjectAgentProvider`.
- **Ορίσετε ένα εργαλείο** χρησιμοποιώντας το διακοσμητή `@tool` ώστε ο πράκτορας να μπορεί να καλεί τις λειτουργίες Python σας.
- **Τρέξετε τον πράκτορα** με ένα μήνυμα χρήστη και να εκτυπώσετε την απάντησή του.
- **Μεταδώσετε απαντήσεις** για άμεση έξοδο.

Στο επόμενο μάθημα θα εξερευνήσουμε τα πλαίσια agentic πιο σε βάθος και θα μάθουμε πώς να δώσουμε στους πράκτορες πιο ισχυρά εργαλεία και δυνατότητες πολυβηματικής λογικής.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση Ευθύνης**:  
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης AI [Co-op Translator](https://github.com/Azure/co-op-translator). Παρόλο που επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη γλώσσα του πρέπει να θεωρείται ως η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
